In [1]:
import os
import base64
import json
import random
from openai import OpenAI
import anthropic
import numpy as np
import re
from tqdm import tqdm
import pandas as pd
import time
from word2number import w2n
from dotenv import load_dotenv


# Ablation Study Configuration

In [2]:
# Ablation Study Configuration
# Choose which agents to enable/disable for the ablation study
ENABLE_VISUAL_AGENT = False   
ENABLE_LANGUAGE_AGENT = True 
ENABLE_HALLUCINATION_AGENT = False 

# Output file suffix for the ablation configuration
ablation_config = []
if ENABLE_VISUAL_AGENT:
    ablation_config.append("visual")
if ENABLE_LANGUAGE_AGENT:
    ablation_config.append("language")
if ENABLE_HALLUCINATION_AGENT:
    ablation_config.append("hallucination")

# Create name suffix based on enabled agents
config_suffix = "_".join(ablation_config)
print(f"Running with configuration: {config_suffix}")

Running with configuration: language


# Load dataset

In [3]:
# Set base directory using relative path
base_dir = os.path.join(os.getcwd(), "dataset", "simpsons")

# Set paths relative to base_dir
annotation_path = os.path.join(base_dir, "v1_Annotation_Val_simpsons_vqa.json")
question_path = os.path.join(base_dir, "v1_Question_Val_simpsons_vqa.json")
images_dir = os.path.join(base_dir, "val_images")

def load_dataset(annotation_path, question_path):
    try:
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']

        with open(question_path, 'r') as f:
            questions = json.load(f)['questions']

        # Select high-quality QA pairs (overall_scores == 1.0)
        filtered_annotations = [
            annotation for annotation in annotations
            if annotation.get('overall_scores', {}).get('question') == 1.0 and
               annotation.get('overall_scores', {}).get('answer') == 1.0
        ]

        # Create a mapping from question ID to answer
        question_id_to_answer = {
            annotation['id']: annotation['answer']
            for annotation in filtered_annotations
        }

        # Create a mapping from question ID to answer type
        question_id_to_answer_type = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation.get('answer_type', 'other')
            }
            for annotation in filtered_annotations
        }

        filtered_questions = [question for question in questions if question['id'] in question_id_to_answer]

        return filtered_questions, filtered_annotations, question_id_to_answer, question_id_to_answer_type

    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], [], {}, {}

def get_dataset(questions, question_id_to_answer, fraction=0.005, seed=42):
    # TODO：Increase quantity
# def get_dataset(questions, question_id_to_answer, fraction=0.05, seed=42):
    try:
        random.seed(seed)
        sample_size = max(1, int(len(questions) * fraction))
        sampled_question_items = random.sample(questions, sample_size)
        sampled_correct_answers = [
            question_id_to_answer[q['id']]
            for q in sampled_question_items
        ]
        # Add image_base64 to each sampled question
        for q in sampled_question_items:
            image_relative_path = q['img_path']
            image_path = os.path.join(images_dir, image_relative_path)
            q['image_base64'] = encode_image(image_path)

        return sampled_question_items, sampled_correct_answers

    except Exception as e:
        print(f"Error sampling dataset: {e}")
        return [], []

def encode_image(image_path):
    try:
        if not os.path.exists(image_path):
            print(f"Error: The image file at {image_path} was not found.")
            return None

        with open(image_path, "rb") as image_file:
            image_base64 = base64.b64encode(image_file.read()).decode('utf-8')
            return image_base64

    except Exception as e:
        print(f"An error occurred while encoding the image: {e}")
        return None

# Initialize results list
results_ablation = []

# Agents Configuration

In [ ]:
load_dotenv()
# Configuration
# MODEL_NAME = "claude-3-5-haiku-20241022"
MODEL_NAME = "gpt-4o-mini"

is_openai_model = not MODEL_NAME.startswith("claude-")

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Agent Implementations

# Enhanced Visual agent that provides highly accurate and question-relevant descriptions
def visual_agent(image_base64, question, max_retries=3, retry_delay=2):
    # If visual agent is disabled, return a basic placeholder
    if not ENABLE_VISUAL_AGENT:
        return "This is a cartoon image from The Simpsons."

    prompt = f"""
    As a visual analysis expert, carefully analyze the image and provide a concise visual description. Avoid speculation or assumptions beyond the visible content. 
    Focus on the following aspects, as relevant to answering the question: {question}.

    1. Characters: Identify characters with distinctive features. 
    2. Actions and Interactions: Describe what character is doing, including body posture and interactions.
    3. Facial Expressions and Emotions: Note visible facial expressions (e.g., happy, surprised, angry).
    4. Scene: Identify whether the scene is indoors or outdoors, and specify the environment.
    5. Objects: Mention relevant items, positions, colors, and sizes.
    6. Layout: Describe where characters and objects are located (e.g., left of, behind).
    7. Attributes and Colors: List visible colors and give exact counts where possible.
    8. Counts: Number of characters or repeated items.
    9. Movement: Describe motion or visual cues if any.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url",
                                 "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                            ],
                        }
                    ],
                    max_tokens=1000,  
                    temperature=0.1,
                )
                visual_desc = completion.choices[0].message.content.strip()
                return visual_desc
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {"type": "base64", "media_type": "image/jpeg", "data": image_base64}},
                        ]
                    }],
                    max_tokens=1000,  
                    temperature=0.1,
                )
                visual_desc = completion.content[0].text.strip()
                return visual_desc

        except Exception as e:
            print(f"Visual agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Visual agent failed to process the image")
    return None

# Language agent: handles text-related tasks, and outputs initial predicted answer
def language_agent(question, image_base64, visual_desc, max_retries=3, retry_delay=2):
    # If language agent is disabled, return None
    if not ENABLE_LANGUAGE_AGENT:
        return None
    # Note: visual_desc is always available regardless of whether visual agent is enabled or not
    # - If visual agent is enabled: visual_desc contains the generated description
    # - If visual agent is disabled: visual_desc contains "This is a cartoon image from The Simpsons."
    prompt = f"""
    As a cartoon language expert, answer the question based on the provided context using EXACTLY ONE WORD:

    Input:
    Question: {question}
    Visual Description: {visual_desc}

    Guidelines: No explanations or punctuation allowed.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                                }
                            ],
                        }
                    ],
                    max_tokens=150,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.choices[0].message.content.strip().lower()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {"type": "base64", "media_type": "image/jpeg", "data": image_base64}},
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.content[0].text.strip().lower()

            # Process the response
            # 1. Remove punctuation marks
            initial_predicted_answer = initial_predicted_answer.rstrip('.!?')

            # 2. Split into words and get the first word
            words = initial_predicted_answer.split()
            if not words:
                continue

            initial_predicted_answer = words[0]

            # 3. Convert numbers if applicable
            try:
                # Check if word represents a number
                number = w2n.word_to_num(initial_predicted_answer)
                initial_predicted_answer = str(number)
            except ValueError:
                # Check if contains numeric digits
                matches = re.findall(r'\d+', initial_predicted_answer)
                if matches:
                    initial_predicted_answer = matches[0]

            return initial_predicted_answer

        except Exception as e:
            print(f"Language agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Language agent failed to generate an answer")
    return None

# Hallucination detection agent: implements a critic without verbose output
def hallucination_agent(question, image_base64, initial_predicted_answer, visual_desc, max_retries=3, retry_delay=2):
    # If hallucination agent is disabled, return the initial prediction
    if not ENABLE_HALLUCINATION_AGENT:
        return initial_predicted_answer
    
    # Handle case when language_agent failed or was disabled
    if initial_predicted_answer is None:
        return "unknown"
    
    # First round: Critic evaluates the initial answer
    prompt_first_round = f"""
    As an expert cartoon visual critic, evaluate the accuracy of the initial answer to this visual question.
    
    Question: {question}
    Initial Answer: {initial_predicted_answer}
    Visual Description: {visual_desc}
    
    EVALUATION FRAMEWORK:
    
    1. UNDERSTAND THE QUESTION REQUIREMENTS
    2. EXAMINE VISUAL EVIDENCE
    3. EVALUATE THE ANSWER'S ACCURACY using this scale:
       - 1.0: Completely correct answer that addresses the question perfectly
       - 0.75: Mostly correct with only minor differences
       - 0.5: Partially correct - contains some correct elements but misses important aspects
       - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
       - 0.0: Completely incorrect
    
    YOUR RESPONSE MUST USE THIS FORMAT:
    ACCURACY: [1.0/0.75/0.5/0.25/0.0]
    EXPLANATION: [brief explanation of why the answer is appropriate or not]
    RECOMMENDATION: [KEEP/REVISE]
    CONFIDENCE: [HIGH/LOW] (HIGH if accuracy is 0.75 or higher, LOW otherwise)
    CORRECTION: [suggest a better answer if recommendation is REVISE, otherwise write 'none']
    """
    
    try:
        if is_openai_model:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_first_round},
                        {"type": "image_url", 
                         "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                    ]
                }],
                max_tokens=300,
                temperature=0.1,
            )
            first_response = completion.choices[0].message.content.strip()
        else:
            completion = client.messages.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_first_round},
                        {"type": "image",
                         "source": {
                             "type": "base64",
                             "media_type": "image/jpeg",
                             "data": image_base64
                         }}
                    ]
                }],
                max_tokens=300,
                temperature=0.1,
            )
            first_response = completion.content[0].text.strip()
        
        # Parse the critic's evaluation silently without printing details
        accuracy_match = re.search(r'ACCURACY:\s*(1\.0|0\.75|0\.5|0\.25|0\.0)', first_response, re.IGNORECASE)
        explanation_match = re.search(r'EXPLANATION:\s*(.+?)(?=\n(RECOMMENDATION|CONFIDENCE|CORRECTION)|$)', first_response, re.IGNORECASE | re.DOTALL)
        recommendation_match = re.search(r'RECOMMENDATION:\s*(KEEP|REVISE)', first_response, re.IGNORECASE)
        confidence_match = re.search(r'CONFIDENCE:\s*(HIGH|LOW)', first_response, re.IGNORECASE)
        correction_match = re.search(r'CORRECTION:\s*(.+?)(?=\n|$)', first_response, re.IGNORECASE | re.DOTALL)
        
        # Extract values with fallbacks
        accuracy = 0.5 if not accuracy_match else float(accuracy_match.group(1))
        explanation = "No explanation provided" if not explanation_match else explanation_match.group(1).strip()
        recommendation = "KEEP" if not recommendation_match else recommendation_match.group(1).upper()
        confidence = "LOW" if accuracy < 0.75 else "HIGH"
        if confidence_match:
            confidence = confidence_match.group(1).upper()
        correction = initial_predicted_answer if not correction_match or correction_match.group(1).strip().lower() == "none" else correction_match.group(1).strip()
        
        # If recommendation is KEEP, just return the initial answer with confidence marker if needed
        if recommendation == "KEEP":
            if confidence == "LOW":
                return f"[LOW CONFIDENCE] {initial_predicted_answer.lower().strip().rstrip('.!?')}"
            return initial_predicted_answer.lower().strip().rstrip('.!?')
        
        # If recommendation is REVISE, go for second round evaluation
        prompt_second_round = f"""
        As an expert cartoon visual validator, make a final decision about the answer to this question.
        
        Question: {question}
        Initial Answer: {initial_predicted_answer}
        Critic's Assessment: {explanation}
        Suggested Correction: {correction}
        
        Based on the image, provide the MOST ACCURATE single-word answer.
        
        YOUR RESPONSE FORMAT:
        FINAL_ANSWER: [single word answer]
        CONFIDENCE: [HIGH/LOW]
        """
        
        if is_openai_model:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_second_round},
                        {"type": "image_url", 
                         "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                    ]
                }],
                max_tokens=150,
                temperature=0.1,
            )
            second_response = completion.choices[0].message.content.strip()
        else:
            completion = client.messages.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_second_round},
                        {"type": "image",
                         "source": {
                             "type": "base64",
                             "media_type": "image/jpeg",
                             "data": image_base64
                         }}
                    ]
                }],
                max_tokens=150,
                temperature=0.1,
            )
            second_response = completion.content[0].text.strip()
        
        # Parse the second response silently
        final_answer_match = re.search(r'FINAL_ANSWER:\s*(.+?)(?=\n|$)', second_response, re.IGNORECASE)
        final_confidence_match = re.search(r'CONFIDENCE:\s*(HIGH/LOW)', second_response, re.IGNORECASE)
        
        final_answer = correction if not final_answer_match else final_answer_match.group(1).strip()
        final_confidence = confidence if not final_confidence_match else final_confidence_match.group(1).upper()
        
        # Format and return the final answer
        final_answer = final_answer.lower().strip().rstrip('.!?')
        if final_confidence == "LOW":
            return f"[LOW CONFIDENCE] {final_answer}"
        return final_answer
        
    except Exception as e:
        # Silent error handling, just write to log without detailed output
        print(f"Hallucination agent error: {str(e)[:100]}..." if len(str(e)) > 100 else f"Hallucination agent error: {e}")
        # Fallback to initial answer with low confidence marker
        return f"[LOW CONFIDENCE] {initial_predicted_answer.lower().strip().rstrip('.!?')}"

Using OpenAI model: gpt-4o-mini


# Calculate accuracy

In [ ]:
def compute_accuracy(question, correct_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2, num_evaluations=3):
    if correct_answer.lower().strip() == predicted_answer.lower().strip():
        return 1.0, [1.0] * num_evaluations

    if correct_answer.lower().strip() + 's' == predicted_answer.lower().strip() or predicted_answer.lower().strip() + 's' == correct_answer.lower().strip():
        return 0.75, [0.75] * num_evaluations
    
    scores = []
    for evaluation_attempt in range(num_evaluations):
        prompt = f"""
        Evaluate the accuracy of the predicted answer according to strict criteria below.

        Input:
        Question: {question}
        Answer type: {answer_type}
        Correct answer: {correct_answer}
        Predicted answer: {predicted_answer}
        
        Evaluation Rules:
        1. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].
        2. Answer Type Considerations:
        - Yes/No: Must be exactly correct (1.0) or wrong (0.0)
        - Number: Must be exactly correct (1.0) or wrong (0.0)
        - Other: Focus PRIMARILY on semantic similarity:

        Scoring Criteria:
        - 1.0: Contains the correct core information, even if phrased differently or with additional details
        - 0.75: Mostly correct but missing minor information or containing slight inaccuracies
        - 0.5: Partially correct - contains some correct elements but misses important aspects
        - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
        - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering
        """
        
        for attempt in range(max_retries):
            try:
                if is_openai_model:
                    completion = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=[{"role": "user", "content": prompt}],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.choices[0].message.content.strip()
                else:
                    completion = client.messages.create(
                        model=MODEL_NAME,
                        messages=[{
                            "role": "user",
                            "content": prompt
                        }],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.content[0].text.strip()

                numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
                if numeric_match:
                    score = float(numeric_match.group(1))
                else:
                    score = 0.0

                scores.append(score)
                break

            except Exception as e:
                print(f"Evaluation attempt {evaluation_attempt+1}, retry {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                # If all retries for this evaluation fail, continue to next evaluation

    # If all evaluations failed, return 0
    if not scores:
        return 0.0, []
        
    # Calculate the result using majority voting
    from collections import Counter
    vote_counter = Counter(scores)
    majority_score, count = vote_counter.most_common(1)[0]  # Get most common score
    
    # If there's a tie, calculate average of the tied values
    if len(scores) > 2:  # Only check for ties with more than 2 scores
        top_scores = vote_counter.most_common()
        if len(top_scores) > 1 and top_scores[0][1] == top_scores[1][1]:  # If there's a tie
            # Find all scores with the same count
            tied_scores = [score for score, count in top_scores if count == top_scores[0][1]]
            majority_score = sum(tied_scores) / len(tied_scores)  # Average of tied scores
    
    return majority_score, scores

# Evaluate model performance

In [ ]:
try:
    # Initialize counters
    correct_count = 0
    total_count = 0
    # Initialize results storage
    accuracies = []

    # Load dataset
    questions, annotations, question_id_to_answer, question_id_to_answer_type = load_dataset(annotation_path, question_path)

    if not questions:
        print("The 'questions' list is empty or not a list.")
        raise ValueError("Questions list is empty")

    # Get sample data
    sampled_questions, sampled_correct_answers = get_dataset(questions, question_id_to_answer)

    if not sampled_questions:
        print("Failed to sample questions or empty sample")
        raise ValueError("No sampled questions")

    # Create an empty collection to store the processed problem IDs
    processed_question_ids = set() 

    # Process each question
    for i, (question_item, correct_answer) in enumerate(tqdm(zip(sampled_questions, sampled_correct_answers),
                                     total=len(sampled_questions))):
        try:
            question_id = question_item['id']
            # Skip if already processed this question
            if question_id in processed_question_ids:
                continue
                
            processed_question_ids.add(question_id)
            
            question = question_item['question']
            image_relative_path = question_item['img_path']
            answer_type = question_id_to_answer_type[question_id]['answer_type']

            print(f"\nProcessing question {i + 1}/{len(sampled_questions)}: ID {question_id}")

            # Build image path and encode
            image_path = os.path.join(images_dir, image_relative_path)
            image_base64 = encode_image(image_path)

            if image_base64 is None:
                print(f"Skipping question ID {question_id} due to image encoding failure")
                continue
            
            # Multi agent processing based on configuration
            # Step 1: Visual agent - now directly including question in the prompt
            visual_desc = visual_agent(image_base64, question=question if ENABLE_VISUAL_AGENT else None)
            if visual_desc is None:
                print(f"Skipping question ID {question_id} - Failed to get image description")
                continue
            
            # Print the visual description when language agent is disabled
            if not ENABLE_LANGUAGE_AGENT:
                print(f"Visual Description: {visual_desc}")

            # Step 2: Language agent processes the question-aware visual description
            initial_predicted_answer = language_agent(question, image_base64, visual_desc)
            
            # If language agent is disabled or failed, use a default answer
            if initial_predicted_answer is None:
                if not ENABLE_LANGUAGE_AGENT:
                    # Default answer when language agent is disabled
                    initial_predicted_answer = "unknown"  
                else:
                    print(f"Skipping question ID {question_id} - Failed to generate answer")
                    continue

            # Step 3: Hallucination agent (may be disabled)
            final_answer = hallucination_agent(
                question=question,
                image_base64=image_base64,
                initial_predicted_answer=initial_predicted_answer,
                visual_desc=visual_desc
            )

            # Use final answer if available, otherwise fallback to initial
            model_answer = final_answer if final_answer else initial_predicted_answer

            # Calculate accuracy
            accuracy, scores = compute_accuracy(
                question=question,
                correct_answer=correct_answer,
                predicted_answer=model_answer,
                answer_type=answer_type
            )

            # Store result
            result = {
                'question_id': question_id,
                'question': question,
                'answer_type': answer_type,
                'correct_answer': correct_answer,
                'predicted_answer': model_answer,
                'evaluator_scores': ", ".join([str(s) for s in scores]),  
                'accuracy': accuracy
            }
            results_ablation.append(result)
            accuracies.append(accuracy)

            # Print results
            print(f"Question ID: {question_id}")
            print(f"Question: {question}")
            print(f"Answer Type: {answer_type}")
            print(f"Correct Answer: {correct_answer}")
            print(f"Predicted Answer: {model_answer}")
            if accuracy is not None:
                accuracies.append(accuracy)
                print(f"Accuracy: {accuracy:.4f}")
            else:
                print(f"Warning: No accuracy for question: {question}")

        except Exception as e:
            print(f"Error processing question {question_item.get('id', 'unknown')}: {e}")
            continue

    # Calculate average accuracy
    if accuracies:
        average_accuracy = np.mean(accuracies)
        print(f"Average Accuracy: {average_accuracy:.4f}")
    else:
        print("No valid accuracy data")
        average_accuracy = 0

except Exception as e:
    print(f"Unexpected error: {e}")
    average_accuracy = 0

  0%|          | 0/36 [00:00<?, ?it/s]


Processing question 1/36: ID 77311


  3%|▎         | 1/36 [00:03<01:48,  3.09s/it]

Question ID: 77311
Question: what is on the shelf?
Answer Type: other
Correct Answer: book
Predicted Answer: books
Accuracy: 0.7500

Processing question 2/36: ID 12809


  6%|▌         | 2/36 [00:04<01:20,  2.36s/it]

...

Question ID: 71583
Question: what is in the background?
Answer Type: other
Correct Answer: table
Predicted Answer: monopoly
Accuracy: 0.0000

Processing question 35/36: ID 37083


 97%|█████████▋| 35/36 [01:14<00:02,  2.43s/it]

Question ID: 37083
Question: what are the men doing?
Answer Type: other
Correct Answer: standing
Predicted Answer: talking
Accuracy: 0.2500

Processing question 36/36: ID 96818


100%|██████████| 36/36 [01:18<00:00,  2.19s/it]

Question ID: 96818
Question: what is the person sitting on?
Answer Type: other
Correct Answer: chair
Predicted Answer: chair
Accuracy: 1.0000
Average Accuracy: 0.7431
Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_gpt_4o_mini.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_gpt_4o_mini.csv


# Save results

In [ ]:
# Clean up ablation results to remove any existing average rows
results_ablation = [r for r in results_ablation if r['question_id'] != 'Average']

# Ensures no duplicate summary rows when saving results
unique_questions = len(set(r['question_id'] for r in results_ablation))

# Add row numbers to each result
for i, result in enumerate(results_ablation, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(results_ablation) + 1,
    'question_id': 'Average',
    'question': f'Total Questions: {unique_questions}',  
    'answer_type': 'All',  
    'correct_answer': '',
    'predicted_answer': '',
    'evaluator_scores': '',  
    'accuracy': average_accuracy 
}
results_ablation.append(average_result)

# Define column order with row_num first
column_order = [
    'row_num',
    'question_id',
    'question',
    'answer_type',
    'correct_answer',
    'predicted_answer',
    'evaluator_scores',  
    'accuracy'
]

# Save results with configuration in filename using the new directory structure
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)

# Create ablation subdirectory if it doesn't exist (using consistent pattern)
os.makedirs(os.path.join(results_dir, "ablation"), exist_ok=True)

# Save to ablation subdirectory with configuration in filename (using consistent pattern)
output_path = os.path.join(results_dir, "ablation", f'simpsons_ablation_{config_suffix}_{safe_model_name}.csv')

# Check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(results_ablation)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_gpt_4o_mini.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_gpt_4o_mini.csv
